In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

spark = SparkSession.builder.appName("Tugas5").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

#langkah 1:data transaksi
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]

data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}

df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi_t5))
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

#langkah 2: membuat df_target
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

print("Siap untuk digunakan dalam tugas")

Siap untuk digunakan dalam tugas


In [12]:
#Soal 1
#ringkasan total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

#menhhitung pencapaian persen
hasil = ringkasan_kota.join(df_target, on="kota", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        34500000|      30000000|     Fitri|114.99999999999999|
|      Solo|        41200000|      40000000|      Bayu|             103.0|
|Yogyakarta|        49175000|      60000000|      Joko| 81.95833333333333|
|  Magelang|        35575000|      45000000|      Rani| 79.05555555555556|
|  Semarang|        36250000|      55000000|      Sari|  65.9090909090909|
+----------+----------------+--------------+----------+------------------+



In [16]:
#Soal 2
ringkasan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

window_kota = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

hasil = ringkasan_kategori.withColumn("row_num", row_number().over(window_kota)) \
    .filter(col("row_num") == 1) \
    .orderBy("kota") \
    .select("kota", "kategori", "total_pendapatan_kategori")

hasil.show()

[Stage 29:=============================>                            (1 + 1) / 2]

+----------+-----------------+-------------------------+
|      kota|         kategori|total_pendapatan_kategori|
+----------+-----------------+-------------------------+
|  Magelang|       Elektronik|                 12625000|
| Purworejo|Makanan & Minuman|                 12775000|
|  Semarang|Makanan & Minuman|                 10300000|
|      Solo|Makanan & Minuman|                 12875000|
|Yogyakarta|       Elektronik|                 15225000|
+----------+-----------------+-------------------------+



In [15]:
#Soal 3
hasil = spark.sql(
    "select t.kota, tg.pic_cabang, count(*) as jumlah_transaksi "
    "from transaksi t "
    "join target tg on t.kota = tg.kota "
    "group by t.kota, tg.pic_cabang "
    "order by jumlah_transaksi desc"
)
hasil.show()

[Stage 25:>                                                         (0 + 2) / 2]

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
|Yogyakarta|      Joko|             109|
|      Solo|      Bayu|             104|
| Purworejo|     Fitri|             101|
|  Magelang|      Rani|              93|
|  Semarang|      Sari|              93|
+----------+----------+----------------+



#soal 5
Dalam hasil A, terlihat bahwa Purworejo berhasil mencapai angka sekitar 114,99%. Ini melampaui sasaran yang seharusnya mencapai 30.000.000. Setelah itu, pada soal B, setiap kota memilik jenis produk yang paling laris sendiri, misalnya Magelang dan Yogyakarta masuk dalam kategori elektronik. Di sisi lain, Purworejo, Semarang, dan Solo termasuk dalam kategori makanan dan minuman. Dari tempat ini bisa terlihat bahwa setiap kota memiliki kebutuhan dan selera pasar yang berbeda. Dalam soal C, terlihat dengan jelas bahwa Solo memiliki jumlah transaksi paling banyak, yaitu sekitar 109 transaksi yang dilakukan oleh Joko. Susun ulang kalimat Dari semua data yang diperoleh, Purworejo dan Solo menunjukkan hasil yang bagus, sedangkan Semarang perlu diperhatikan lebih karena sering menempati peringkat terendah.

In [35]:
# Right Join - kebalikan dari left join, semua baris dari df_produk dipertahankan
df_right = df_transaksi.join(df_target, on="kota", how="right")
df_right.show(3)

# Full Outer Join - semua baris dari kedua tabel dipertahankan, meskipun ngga match
df_full = df_transaksi.join(df_target, on="kota", how="full")
df_full.show(3)

# Left Semi Join - cuma menampilkan kolom dari tabel kiri yang punya pasangan di tabel kanan
df_semi = df_transaksi.join(df_target, on="kota", how="left_semi")
df_semi.show(3)

# Left Anti Join - kebalikan dari semi join, menampilkan baris kiri yang TIDAK punya pasangan
df_anti = df_transaksi.join(df_target, on="kota", how="left_anti")
df_anti.show(3)

+--------+--------+--------------------+------------+------------+----------+--------------+----------+
|    kota|order_id|            kategori|unit_terjual|harga_satuan|pendapatan|target_bulanan|pic_cabang|
+--------+--------+--------------------+------------+------------+----------+--------------+----------+
|Magelang| TRX-497|          Elektronik|           3|       75000|    225000|      45000000|      Rani|
|Magelang| TRX-487|Kesehatan & Kecan...|           7|       50000|    350000|      45000000|      Rani|
|Magelang| TRX-482|          Elektronik|           3|      100000|    300000|      45000000|      Rani|
+--------+--------+--------------------+------------+------------+----------+--------------+----------+
only showing top 3 rows



+--------+--------+--------------------+------------+------------+----------+--------------+----------+
|    kota|order_id|            kategori|unit_terjual|harga_satuan|pendapatan|target_bulanan|pic_cabang|
+--------+--------+--------------------+------------+------------+----------+--------------+----------+
|Magelang|   TRX-8|Kesehatan & Kecan...|           5|       75000|    375000|      45000000|      Rani|
|Magelang|   TRX-9|             Fashion|           8|       25000|    200000|      45000000|      Rani|
|Magelang|  TRX-11|             Fashion|           5|      100000|    500000|      45000000|      Rani|
+--------+--------+--------------------+------------+------------+----------+--------------+----------+
only showing top 3 rows



+--------+--------+--------------------+------------+------------+----------+
|    kota|order_id|            kategori|unit_terjual|harga_satuan|pendapatan|
+--------+--------+--------------------+------------+------------+----------+
|Magelang|   TRX-8|Kesehatan & Kecan...|           5|       75000|    375000|
|Magelang|   TRX-9|             Fashion|           8|       25000|    200000|
|Magelang|  TRX-11|             Fashion|           5|      100000|    500000|
+--------+--------+--------------------+------------+------------+----------+
only showing top 3 rows



[Stage 193:============================>                            (1 + 1) / 2]

+----+--------+--------+------------+------------+----------+
|kota|order_id|kategori|unit_terjual|harga_satuan|pendapatan|
+----+--------+--------+------------+------------+----------+
+----+--------+--------+------------+------------+----------+



In [23]:
from pyspark.sql.functions import dense_rank, lag, lead, sum as spark_sum

window_spec = Window.partitionBy("kota").orderBy(col("pendapatan").desc())

# dense_rank() - mirip rank(), tapi TIDAK ada celah nomor setelah nilai yang sama
df_dense = df_transaksi.withColumn("dense_peringkat", dense_rank().over(window_spec))
df_dense.show(3)

# lag() - ambil nilai dari baris SEBELUMNYA dalam partisi
df_lag = df_transaksi.withColumn("pendapatan_sebelumnya", lag("pendapatan", 1).over(window_spec))
df_lag.show(3)

# lead() - ambil nilai dari baris SESUDAHNYA dalam partisi
df_lead = df_transaksi.withColumn("pendapatan_berikutnya", lead("pendapatan", 1).over(window_spec))
df_lead.show(3)

# running total (total kumulatif) pakai window tanpa partitionBy
window_running = Window.orderBy("order_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_running = df_transaksi.withColumn("total_kumulatif", spark_sum("pendapatan").over(window_running))
df_running.show(3)

+--------+-----------------+--------+------------+------------+----------+---------------+
|order_id|         kategori|    kota|unit_terjual|harga_satuan|pendapatan|dense_peringkat|
+--------+-----------------+--------+------------+------------+----------+---------------+
| TRX-268|          Fashion|Magelang|           9|      150000|   1350000|              1|
| TRX-342|       Elektronik|Magelang|           8|      150000|   1200000|              2|
| TRX-199|Makanan & Minuman|Magelang|           7|      150000|   1050000|              3|
+--------+-----------------+--------+------------+------------+----------+---------------+
only showing top 3 rows

+--------+-----------------+--------+------------+------------+----------+---------------------+
|order_id|         kategori|    kota|unit_terjual|harga_satuan|pendapatan|pendapatan_sebelumnya|
+--------+-----------------+--------+------------+------------+----------+---------------------+
| TRX-268|          Fashion|Magelang|          

In [26]:
# CASE WHEN di Spark SQL - bikin kategori baru berdasarkan kondisi
hasil_case = spark.sql(
    "select order_id, kota, pendapatan, "
    "case when pendapatan >= 300000 then 'Tinggi' "
    "     when pendapatan >= 150000 then 'Sedang' "
    "     else 'Rendah' end as level_pendapatan "
    "from transaksi"
)
hasil_case.show(5)

# Subquery - kueri di dalam kueri
hasil_subquery = spark.sql(
    "select kota, avg_pendapatan from "
    "(select kota, avg(pendapatan) as avg_pendapatan from transaksi group by kota) as ringkasan "
    "where avg_pendapatan > 200000"
)
hasil_subquery.show()

# Window function langsung di dalam Spark SQL (pakai OVER, PARTITION BY)
hasil_window_sql = spark.sql(
    "select kota, order_id, pendapatan, "
    "row_number() over (partition by kota order by pendapatan desc) as peringkat "
    "from transaksi"
)
hasil_window_sql.filter(col("peringkat") == 1).show()

+--------+----------+----------+----------------+
|order_id|      kota|pendapatan|level_pendapatan|
+--------+----------+----------+----------------+
|   TRX-0|      Solo|    125000|          Rendah|
|   TRX-1| Purworejo|     75000|          Rendah|
|   TRX-2|Yogyakarta|    400000|          Tinggi|
|   TRX-3|Yogyakarta|    150000|          Sedang|
|   TRX-4| Purworejo|    250000|          Sedang|
+--------+----------+----------+----------------+
only showing top 5 rows



+----------+------------------+
|      kota|    avg_pendapatan|
+----------+------------------+
|  Magelang| 382526.8817204301|
|  Semarang|389784.94623655913|
|      Solo| 396153.8461538461|
| Purworejo| 341584.1584158416|
|Yogyakarta| 451146.7889908257|
+----------+------------------+

+----------+--------+----------+---------+
|      kota|order_id|pendapatan|peringkat|
+----------+--------+----------+---------+
|  Magelang| TRX-268|   1350000|        1|
| Purworejo|  TRX-24|    900000|        1|
|  Semarang| TRX-152|   1350000|        1|
|      Solo| TRX-106|   1350000|        1|
|Yogyakarta| TRX-246|   1350000|        1|
+----------+--------+----------+---------+



In [30]:
from pyspark.sql.functions import broadcast

# Broadcast join - dipakai kalau salah satu tabel ukurannya kecil (misal df_produk/df_target)
# Spark akan copy tabel kecil ke semua node, jadi gak perlu shuffle data besar
df_broadcast = df_transaksi.join(broadcast(df_target), on="kota", how="inner")
df_broadcast.show(5)

# Cek execution plan buat lihat broadcast beneran kepake atau nggak
df_broadcast.explain()

+----------+--------+--------------------+------------+------------+----------+--------------+----------+
|      kota|order_id|            kategori|unit_terjual|harga_satuan|pendapatan|target_bulanan|pic_cabang|
+----------+--------+--------------------+------------+------------+----------+--------------+----------+
|      Solo|   TRX-0|             Fashion|           5|       25000|    125000|      40000000|      Bayu|
| Purworejo|   TRX-1|   Makanan & Minuman|           1|       75000|     75000|      30000000|     Fitri|
|Yogyakarta|   TRX-2|Kesehatan & Kecan...|           8|       50000|    400000|      60000000|      Joko|
|Yogyakarta|   TRX-3|          Elektronik|           3|       50000|    150000|      60000000|      Joko|
| Purworejo|   TRX-4|             Fashion|           5|       50000|    250000|      30000000|     Fitri|
+----------+--------+--------------------+------------+------------+----------+--------------+----------+
only showing top 5 rows

== Physical Plan ==
A

In [31]:
# Pivot - misal mau lihat total pendapatan per kota, dipecah per kategori jadi kolom
df_pivot = df_transaksi.groupBy("kota").pivot("kategori").sum("pendapatan")
df_pivot.show()

# Pivot dengan kolom kategori dibatasi manual (lebih cepat performanya)
df_pivot2 = df_transaksi.groupBy("kota").pivot(
    "kategori", ["Elektronik", "Fashion", "Makanan & Minuman"]
).sum("pendapatan")
df_pivot2.show()

+----------+----------+--------+----------------------+-----------------+
|      kota|Elektronik| Fashion|Kesehatan & Kecantikan|Makanan & Minuman|
+----------+----------+--------+----------------------+-----------------+
|  Magelang|  12625000|10000000|               3675000|          9275000|
|  Semarang|   7250000| 9275000|               9425000|         10300000|
|      Solo|   5050000|10425000|              12850000|         12875000|
| Purworejo|   7650000| 9125000|               4950000|         12775000|
|Yogyakarta|  15225000| 7350000|              13650000|         12950000|
+----------+----------+--------+----------------------+-----------------+



[Stage 164:============================>                            (1 + 1) / 2]

+----------+----------+--------+-----------------+
|      kota|Elektronik| Fashion|Makanan & Minuman|
+----------+----------+--------+-----------------+
|  Magelang|  12625000|10000000|          9275000|
|  Semarang|   7250000| 9275000|         10300000|
|      Solo|   5050000|10425000|         12875000|
| Purworejo|   7650000| 9125000|         12775000|
|Yogyakarta|  15225000| 7350000|         12950000|
+----------+----------+--------+-----------------+



In [33]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

# Bikin fungsi Python biasa
def kategori_pendapatan(nilai):
    if nilai >= 300000:
        return "Tinggi"
    elif nilai >= 150000:
        return "Sedang"
    else:
        return "Rendah"

# Daftarkan sebagai UDF supaya bisa dipakai di kolom Spark
udf_kategori = udf(kategori_pendapatan, StringType())

df_udf = df_transaksi.withColumn("level_pendapatan", udf_kategori(col("pendapatan")))
df_udf.show(5)

# Catatan: UDF lebih lambat dibanding fungsi bawaan Spark (spark_sum, when, dll)
# karena data harus "keluar-masuk" antara JVM Spark dan proses Python

[Stage 171:>                                                        (0 + 1) / 1]

+--------+--------------------+----------+------------+------------+----------+----------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|level_pendapatan|
+--------+--------------------+----------+------------+------------+----------+----------------+
|   TRX-0|             Fashion|      Solo|           5|       25000|    125000|          Rendah|
|   TRX-1|   Makanan & Minuman| Purworejo|           1|       75000|     75000|          Rendah|
|   TRX-2|Kesehatan & Kecan...|Yogyakarta|           8|       50000|    400000|          Tinggi|
|   TRX-3|          Elektronik|Yogyakarta|           3|       50000|    150000|          Sedang|
|   TRX-4|             Fashion| Purworejo|           5|       50000|    250000|          Sedang|
+--------+--------------------+----------+------------+------------+----------+----------------+
only showing top 5 rows



In [34]:
# Kalau df_transaksi dipakai berkali-kali, cache biar gak dihitung ulang dari awal tiap kali dipanggil
df_transaksi.cache()
df_transaksi.count()  # trigger biar cache-nya beneran kesimpen

# Cek apakah DataFrame sudah di-cache
print(df_transaksi.storageLevel)

# Hapus cache kalau sudah gak dipakai lagi, biar memori lega
df_transaksi.unpersist()

[Stage 172:>                                                        (0 + 2) / 2]

Disk Memory Deserialized 1x Replicated


DataFrame[order_id: string, kategori: string, kota: string, unit_terjual: bigint, harga_satuan: bigint, pendapatan: bigint]